# 03 — Database Verification

Confirm `fetch_historical.py` loaded the historical candles correctly:
1. Row count & date range
2. Per-symbol counts
3. **Gap check** — are any candles missing?
4. Expected vs actual count

In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))  # so `src` is importable from notebooks/

import pandas as pd
from src.storage.postgres import engine
from src.params.constants import DEFAULT_REST_INTERVAL, SUPPORTED_SYMBOLS

SYMBOL = SUPPORTED_SYMBOLS[0].value        # "BTCUSDT"

# Convert the configured interval (e.g. "15m", "1h", "1d") to minutes
_interval = DEFAULT_REST_INTERVAL.value
INTERVAL_MINUTES = int(_interval[:-1]) * {"m": 1, "h": 60, "d": 1440}[_interval[-1]]

print(f"Symbol: {SYMBOL} | Interval: {_interval} ({INTERVAL_MINUTES} min)")

Symbol: BTCUSDT | Interval: 1m (1 min)


## 1. Row count & date range

In [2]:
pd.read_sql(
    'SELECT count(*) AS rows, min(open_time) AS first, max(open_time) AS last FROM ohlcv_data',
    engine,
)

,rows,first,last
0,239610,2026-01-01,2026-06-16 09:29:00


## 2. Per-symbol counts

In [3]:
pd.read_sql(
    'SELECT symbol, count(*) AS rows, min(open_time) AS first, max(open_time) AS last '
    'FROM ohlcv_data GROUP BY symbol',
    engine,
)

,symbol,rows,first,last
0,BTCUSDT,239610,2026-01-01,2026-06-16 09:29:00


## 3. Per-symbol gap & completeness check
For each symbol: count gaps (consecutive `open_time` differing by more than one interval) and compare expected vs actual candle counts. `missing == 0` means complete.

In [4]:
step = pd.Timedelta(minutes=INTERVAL_MINUTES)
summary = []

for sym in SUPPORTED_SYMBOLS:
    s = sym.value
    df = pd.read_sql(
        'SELECT open_time FROM ohlcv_data WHERE symbol = %(s)s ORDER BY open_time',
        engine,
        params={'s': s},
    )
    if df.empty:
        summary.append({'symbol': s, 'rows': 0, 'expected': 0, 'missing': 0, 'gaps': 0})
        continue
    first, last = df['open_time'].min(), df['open_time'].max()
    expected = int((last - first) / step) + 1
    actual = len(df)
    gaps = int((df['open_time'].diff() > step).sum())
    summary.append({
        'symbol': s, 'first': first, 'last': last,
        'rows': actual, 'expected': expected,
        'missing': expected - actual, 'gaps': gaps,
    })

pd.DataFrame(summary)

,symbol,first,last,rows,expected,missing,gaps
0,BTCUSDT,2026-01-01,2026-06-16 09:29:00,239610,239610,0,0
